In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, classification_report, confusion_matrix
from skimage.metrics import structural_similarity as ssim
from tqdm import tqdm
import zipfile
import urllib.request
import time
import kagglehub  # Import kagglehub

# 1. Download Dataset
# Gunakan link langsung ke dataset.
# dataset_url = "https://github.com/jahanzaibjutt/Brain-Tumor-MRI-Dataset/archive/refs/heads/main.zip"  # URL yang benar
zip_file_name = "/content/brain_tumor_dataset.zip"  # Path yang diberikan oleh pengguna
extract_folder = "brain_tumor_dataset"
max_retries = 3
retry_delay = 5  # detik

# Download dataset
if not os.path.exists(zip_file_name):
    for attempt in range(max_retries):
        try:
            print(f"Mengunduh dataset dari Kaggle menggunakan kagglehub...")
            # Gunakan kagglehub untuk mengunduh dataset
            path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")  # Menghapus argumen force
            print("Dataset berhasil diunduh.")
            break  # Jika pengunduhan berhasil, keluar dari loop
        except Exception as e:
            print(f"Error saat mengunduh dataset (percobaan {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                print(f"Mencoba lagi dalam {retry_delay} detik...")
                time.sleep(retry_delay)
            else:
                print("Gagal mengunduh dataset setelah beberapa kali percobaan.")
                print("Silakan unduh dataset secara manual dan letakkan di direktori yang sama dengan notebook ini.")
                raise  # Munculkan kembali exception untuk menghentikan eksekusi
    # zip_file_name = path + ".zip"  # Perbaiki nama file zip. Kagglehub mengembalikan path ke folder yang diekstrak.
    extract_folder = path  # Path sudah diekstrak oleh kagglehub
else:
    print(f"File zip '{zip_file_name}' sudah ada.")

# Ekstrak dataset
# if not os.path.exists(extract_folder): # Tidak perlu diekstrak, kagglehub mengekstraknya.
#     try:
#         with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
#             zip_ref.extractall(extract_folder)
#         print(f"Dataset diekstrak ke '{extract_folder}'")
#     except zipfile.BadZipFile as e:
#         print(f"Error saat mengekstrak dataset: {e}")
#         print("Pastikan file zip valid dan tidak rusak. Anda mungkin perlu menghapus file tersebut dan mencoba mengunduhnya lagi, atau menyediakan file zip yang valid.")
#         raise  # Munculkan kembali exception untuk menghentikan eksekusi
#     except Exception as e:
#         print(f"Terjadi error tak terduga saat ekstraksi: {e}")
#         raise
# else:
#     print(f"Dataset sudah diekstrak ke '{extract_folder}'")
print(f"Dataset sudah diekstrak ke '{extract_folder}'")  # selalu diekstrak oleh kagglehub

# 2. Load dan Preprocess Data
def load_images_from_folder(folder, label):
    images = []
    labels = []
    for filename in tqdm(os.listdir(folder)):  # Gunakan tqdm untuk indikasi progress
        img_path = os.path.join(folder, filename)
        if os.path.isfile(img_path):
            try:
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img = cv2.resize(img, (128, 128))
                    images.append(img)
                    labels.append(label)
                else:
                    print(f"Peringatan: Tidak dapat membaca gambar {img_path}. Melewati.")
            except Exception as e:
                print(f"Error saat memproses gambar {img_path}: {e}. Melewati.")
                continue  # Lewati ke file berikutnya
    return images, labels

# Sesuaikan path ke tempat data diekstrak.
base_path = os.path.join(extract_folder, "Training")  # Path yang benar
categories = ["glioma", "meningioma", "notumor", "pituitary"]
label_map = {
    0: "glioma",
    1: "meningioma",
    2: "notumor",
    3: "pituitary"
}

images = []
labels = []

for i, category in enumerate(categories):
    folder = os.path.join(base_path, category)
    if not os.path.exists(folder):
        print(f"Folder tidak ditemukan: {folder}")
        continue
    imgs, lbls = load_images_from_folder(folder, i)
    images.extend(imgs)
    labels.extend(lbls)

if not images:
    print("Tidak ada gambar yang dimuat. Harap periksa path dataset dan pastikan gambar dapat dibaca.")
    raise ValueError("Tidak ada gambar yang dimuat")  # hentikan jika tidak ada gambar yang dimuat

images = np.array(images).astype("float32") / 255.0
images = np.expand_dims(images, -1)
labels = np.array(labels)

# 3. Autoencoder
input_img = Input(shape=(128, 128, 1))
x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
encoded = MaxPooling2D((2, 2), padding='same')(x)

x = Conv2D(64, (3, 3), activation='relu', padding='same')(encoded)
x = UpSampling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.summary()

# 4. Train Autoencoder
x_train, x_test = train_test_split(images, test_size=0.2, random_state=42)
autoencoder.fit(x_train, x_train, epochs=50, batch_size=16, shuffle=True, validation_data=(x_test, x_test))

# 5. Evaluasi Autoencoder
decoded_imgs = autoencoder.predict(x_test)

mse = mean_squared_error(x_test.flatten(), decoded_imgs.flatten())
# ssim_total = np.mean([ssim(x_test[i].squeeze(), decoded_imgs[i].squeeze()) for i in range(len(x_test))])
ssim_total = np.mean([ssim(x_test[i].squeeze(), decoded_imgs[i].squeeze(), data_range=1.0) for i in range(len(x_test))])  # Tambahkan data_range
print(f"MSE: {mse}")
print(f"SSIM: {ssim_total}")

# 6. Simpan Autoencoder
autoencoder.save("autoencoder_model.h5")

# 7. CNN Training
x_train_cnn, x_test_cnn, y_train_cnn, y_test_cnn = train_test_split(images, labels, test_size=0.2, random_state=42)
y_train_cat = to_categorical(y_train_cnn, 4)
y_test_cat = to_categorical(y_test_cnn, 4)

cnn = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn.summary()
cnn.fit(x_train_cnn, y_train_cat, epochs=100, batch_size=16, validation_data=(x_test_cnn, y_test_cat))

# 8. Simpan model CNN
cnn.save("cnn_model.h5")

# 9. Evaluasi CNN
test_loss, test_acc = cnn.evaluate(x_test_cnn, y_test_cat)
print(f"Akurasi Tes: {test_acc * 100:.2f}%")

y_pred = cnn.predict(x_test_cnn)
y_pred_classes = np.argmax(y_pred, axis=1)

# 10. Tampilkan Beberapa Gambar dengan Label dan Anomali
plt.figure(figsize=(15, 15))
for i in range(min(25, len(x_test_cnn))):  # Tampilkan maksimal 25 gambar
    plt.subplot(5, 5, i + 1)
    plt.imshow(x_test_cnn[i].squeeze(), cmap='gray')
    true_label = label_map[y_test_cnn[i]]
    pred_label = label_map[y_pred_classes[i]]

    # Hitung MSE untuk menandai anomali
    img_mse = mean_squared_error(x_test_cnn[i].flatten(), decoded_imgs[i].flatten())
    if img_mse > 0.01:  # Sesuaikan threshold sesuai kebutuhan
        plt.title(f"Asli: {true_label}\nPrediksi: {pred_label} (Anomali)")
        plt.xlabel(f"MSE: {img_mse:.4f}", color='red')  # Tambahkan nilai MSE
        for spine in plt.gca().spines.values():
            spine.edgecolor = 'red'  # Ubah warna border
    else:
        plt.title(f"Asli: {true_label}\nPrediksi: {pred_label}")
        plt.xlabel(f"MSE: {img_mse:.4f}", color='green')
    plt.axis('off')
plt.show()

# 11. Confusion Matrix
print("\nLaporan Klasifikasi:")
print(classification_report(y_test_cnn, y_pred_classes, target_names=list(label_map.values())))

cm = confusion_matrix(y_test_cnn, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(label_map.values()),
            yticklabels=list(label_map.values()))
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title("Confusion Matrix")
plt.show()